# Midterm-Style Submission Notebook

This is the only supported submission notebook.
It mirrors the archived midterm evaluation flow from `archive/notebooks/evaluation/DL_Midterm_Eval.ipynb`, but uses the Google notebook setup for Drive mount, repo checkout, merged-model discovery, and persistent output files.

Historical experiment notebooks are preserved in `archive/` for documentation, but they are not supported execution paths.

This next cell mounts Google Drive, clones the repo into the runtime, copies the required CSV inputs into the checkout, and defines the persistent Drive output directory.

- Reads: Google Drive, GitHub repo, runtime filesystem
- Writes: `/content/svg-kaggle-comptetition`, Drive output folders
- Rerun-safe: Yes. It resets the runtime checkout and recopies the inputs.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT_ON_DRIVE = DRIVE_ROOT / "svg-kaggle-comptetition"
OUTPUT_ROOT = PROJECT_ROOT_ON_DRIVE / "submission_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])

for required_name in ["test.csv", "train.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    candidate_paths = [
        PROJECT_ROOT_ON_DRIVE / required_name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / required_name,
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            break
    else:
        raise FileNotFoundError(f"Could not find {required_name} in Google Drive.")

os.chdir(CHECKOUT_PATH)
print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Persistent output root: {OUTPUT_ROOT}")


## Package Check

This next cell installs the runtime packages needed by the midterm-style notebook, using normal Python subprocess calls instead of notebook shell magics.

- Reads: Current Python environment
- Writes: Installed runtime packages
- Rerun-safe: Yes. It only installs packages that are missing.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "trl": "trl",
    "cairosvg": "cairosvg",
    "lxml": "lxml",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


## Imports And Config

This next cell imports the notebook dependencies and defines the same core settings used by the midterm notebook, plus the Drive-backed output paths for this setup.

- Reads: Runtime packages, repo paths
- Writes: In-memory constants and config only
- Rerun-safe: Yes. It only defines notebook state.


In [ ]:
import io
import os
import re
import shutil
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
import pandas as pd
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
from IPython.display import display
from PIL import Image
from lxml import etree
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_PATH = CHECKOUT_PATH / "svg-lora-adapter"
MERGED_PATH = CHECKOUT_PATH / "svg-model-merged"
DRIVE_ADAPTER_PATH = PROJECT_ROOT_ON_DRIVE / "svg-lora-adapter"
DRIVE_MERGED_PATH = PROJECT_ROOT_ON_DRIVE / "svg-model-merged"
MERGED_WEIGHT_FILES = (
    "model.safetensors",
    "model.safetensors.index.json",
    "pytorch_model.bin",
    "pytorch_model.bin.index.json",
)

def has_model_weights(path: Path) -> bool:
    return path.exists() and any((path / name).exists() for name in MERGED_WEIGHT_FILES)

ADAPTER_REQUIRED_FILES = (
    "adapter_config.json",
    "adapter_model.safetensors",
)

def has_adapter_files(path: Path) -> bool:
    return path.exists() and all((path / name).exists() for name in ADAPTER_REQUIRED_FILES)

TEST_CSV = CHECKOUT_PATH / "test.csv"
SUBMISSION_CSV = OUTPUT_ROOT / "submission.csv"
DEBUG_CSV = OUTPUT_ROOT / "submission_debug.csv"

PASS1_MAX_NEW_TOKENS = 2048
PASS2_MAX_NEW_TOKENS = 3072
RETRY_MODE = "long_greedy"  # Options: "off", "long_greedy", "sample"
RETRY_ON_STAGES = ("xml", "fallback")
RETRY_ON_CAPPED = True
SAMPLE_TEMPERATURE = 0.2
SAMPLE_TOP_P = 0.95
SAMPLE_TOP_K = 20
PASS1_BATCH_SIZE = 250
PASS2_BATCH_SIZE = 128
RUN_LIMIT = None  # Set to 10 for a quick smoke test in Colab.

MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
RENDER_SIZE = 256

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter"
}

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256" width="256" height="256">'
    '<rect width="256" height="256" fill="white"/>'
    '</svg>'
)

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)

print(f"Test CSV: {TEST_CSV}")
print(f"Submission output: {SUBMISSION_CSV}")
print(f"Debug output: {DEBUG_CSV}")
print(f"Run limit: {RUN_LIMIT}")
print(f"Pass 1 batch size: {PASS1_BATCH_SIZE}")
print(f"Pass 2 batch size: {PASS2_BATCH_SIZE}")
print(f"Pass 1 max_new_tokens: {PASS1_MAX_NEW_TOKENS}")
print(f"Pass 2 max_new_tokens: {PASS2_MAX_NEW_TOKENS}")


## Load Model

This next cell follows the midterm-style inference path. It prefers merged model weights, copies them from Drive into the runtime checkout if needed, and only falls back to base model plus LoRA if merged weights are still unavailable.

- Reads: Local merged model directory, Drive merged model directory, adapter files, Hugging Face model files
- Writes: In-memory tokenizer and model state only
- Rerun-safe: Yes. It reloads the model path for the current runtime.


In [ ]:
# Historical note: this merged-model autodetect fallback was kept in the archived retry notebook for debugging provenance.
local_merged_ready = has_model_weights(MERGED_PATH)
drive_merged_ready = has_model_weights(DRIVE_MERGED_PATH)
local_adapter_ready = has_adapter_files(ADAPTER_PATH)
drive_adapter_ready = has_adapter_files(DRIVE_ADAPTER_PATH)

if not local_merged_ready and drive_merged_ready:
    if MERGED_PATH.exists():
        shutil.rmtree(MERGED_PATH)
    shutil.copytree(DRIVE_MERGED_PATH, MERGED_PATH)
    local_merged_ready = True
    print(f"Copied merged model from Drive: {DRIVE_MERGED_PATH}")
elif MERGED_PATH.exists() and not local_merged_ready:
    print(f"Merged model directory exists at {MERGED_PATH}, but no weight files were found.")

if not local_adapter_ready and drive_adapter_ready:
    if ADAPTER_PATH.exists():
        shutil.rmtree(ADAPTER_PATH)
    shutil.copytree(DRIVE_ADAPTER_PATH, ADAPTER_PATH)
    local_adapter_ready = True
    print(f"Copied adapter from Drive: {DRIVE_ADAPTER_PATH}")
elif ADAPTER_PATH.exists() and not local_adapter_ready:
    print(f"Adapter directory exists at {ADAPTER_PATH}, but required files were not found.")

if local_merged_ready:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MERGED_PATH,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    MODEL_SOURCE = f"merged model at {MERGED_PATH}"
else:
    if not local_adapter_ready:
        raise FileNotFoundError(
            "Merged model weights were not found, and the LoRA adapter is also unavailable. "
            f"Checked merged paths: {MERGED_PATH} and {DRIVE_MERGED_PATH}. "
            f"Checked adapter paths: {ADAPTER_PATH} and {DRIVE_ADAPTER_PATH}."
        )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH), local_files_only=True)
    MODEL_SOURCE = f"base model + LoRA adapter at {ADAPTER_PATH}"
    print("Merged model not found. Using the temporary base-plus-LoRA fallback.")
    print("Remove this fallback path once svg-model-merged is uploaded to Drive.")

model.eval()
if model.config.pad_token_id is None and tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id

for attr_name in ("temperature", "top_p", "top_k"):
    if hasattr(model.generation_config, attr_name):
        setattr(model.generation_config, attr_name, None)

print(f"Loaded model source: {MODEL_SOURCE}")
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("model pad:", model.config.pad_token_id)


## SVG Helpers

This next cell loads the retry-aware SVG cleanup and validation helper module used by the improved submission path.

- Reads: Raw model text, SVG strings
- Writes: In-memory helper functions only
- Rerun-safe: Yes. It only defines functions.


In [ ]:
import base64
import importlib
import sys

HELPER_PATH = CHECKOUT_PATH / "submission_inference_utils.py"
HELPER_MODULE_B64 = "..."
HELPER_PATH.write_bytes(base64.b64decode(HELPER_MODULE_B64))

if str(CHECKOUT_PATH) not in sys.path:
    sys.path.insert(0, str(CHECKOUT_PATH))

submission_inference_utils = importlib.import_module("submission_inference_utils")
submission_inference_utils = importlib.reload(submission_inference_utils)

InferenceConfig = submission_inference_utils.InferenceConfig
HELPER_VERSION = submission_inference_utils.HELPER_VERSION
build_submission_csv_with_retry = submission_inference_utils.build_submission_csv

INFERENCE_CONFIG = InferenceConfig(
    pass1_max_new_tokens=PASS1_MAX_NEW_TOKENS,
    pass2_max_new_tokens=PASS2_MAX_NEW_TOKENS,
    retry_mode=RETRY_MODE,
    retry_on_stages=RETRY_ON_STAGES,
    retry_on_capped=RETRY_ON_CAPPED,
    sample_temperature=SAMPLE_TEMPERATURE,
    sample_top_p=SAMPLE_TOP_P,
    sample_top_k=SAMPLE_TOP_K,
    pass1_batch_size=PASS1_BATCH_SIZE,
    pass2_batch_size=PASS2_BATCH_SIZE,
    render_size=RENDER_SIZE,
    max_svg_length=MAX_SVG_LENGTH,
    max_path_count=MAX_PATH_COUNT,
    allowed_tags=tuple(sorted(ALLOWED_TAGS)),
    fallback_svg=FALLBACK_SVG,
)

print(f"Loaded helper version: {HELPER_VERSION}")
print(INFERENCE_CONFIG)


## Batched Generation And Submission Build

This next cell wraps the improved two-pass inference builder so the rest of the notebook can keep the same execution flow.

- Reads: Test prompts, tokenizer, model, SVG helper functions
- Writes: In-memory generated SVGs plus the final CSV files when `build_submission_csv` runs
- Rerun-safe: Yes. It only defines functions until the final execution cell calls them.


In [ ]:
def build_submission_csv(
    test_csv_path=TEST_CSV,
    output_csv_path=SUBMISSION_CSV,
    debug_csv_path=DEBUG_CSV,
    pass1_batch_size=PASS1_BATCH_SIZE,
    pass2_batch_size=PASS2_BATCH_SIZE,
    limit=RUN_LIMIT,
):
    return build_submission_csv_with_retry(
        test_csv_path=test_csv_path,
        output_csv_path=output_csv_path,
        debug_csv_path=debug_csv_path,
        model=model,
        tokenizer=tokenizer,
        config=INFERENCE_CONFIG,
        pass1_batch_size=pass1_batch_size,
        pass2_batch_size=pass2_batch_size,
        limit=limit,
    )


## Run Submission Build

This next cell runs the full midterm-style submission build and writes only `submission.csv` and `submission_debug.csv` into the persistent Drive output directory.

- Reads: `test.csv`, tokenizer, model, helper functions
- Writes: `submission.csv`, `submission_debug.csv`
- Rerun-safe: Yes. It overwrites the saved output files.


In [ ]:
submission_df = build_submission_csv(
    test_csv_path=TEST_CSV,
    output_csv_path=SUBMISSION_CSV,
    debug_csv_path=DEBUG_CSV,
    pass1_batch_size=PASS1_BATCH_SIZE,
    pass2_batch_size=PASS2_BATCH_SIZE,
    limit=RUN_LIMIT,
)

print(f"Saved submission to: {SUBMISSION_CSV}")
print(f"Saved debug output to: {DEBUG_CSV}")


## Final Review

This next cell reads the saved output files back from Drive and previews the final submission and debug dataframes.

- Reads: `submission.csv`, `submission_debug.csv`
- Writes: Nothing
- Rerun-safe: Yes. Read-only review cell.


In [ ]:
saved_submission_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
saved_debug_df = pd.read_csv(DEBUG_CSV, keep_default_na=False)

print(f"Saved submission rows: {len(saved_submission_df)}")
print(f"Saved debug rows: {len(saved_debug_df)}")
display(saved_submission_df.head())
display(saved_debug_df.head())
